# [cupuu] - Football Score Prediction


## 1. Setup

In [1]:
import os, gc, json, time, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import poisson, ks_2samp
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import TPESampler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

MASTER_SEED = 42
np.random.seed(MASTER_SEED); random.seed(MASTER_SEED)
os.environ["PYTHONHASHSEED"] = str(MASTER_SEED)

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "dataset"

N_FOLDS              = 5
MAX_GOAL             = 8
N_BAG_SEEDS          = 80      # bagging seed per (model, target)
N_OPTUNA_TRIALS      = {"lgbm_poisson": 300, "lgbm_mae": 300,
                        "xgb_poisson":  250, "catboost_mae": 200}
N_PSEUDO_ROUNDS      = 3
PSEUDO_CONF_QUANTILE = 0.75    # threshold confidence untuk include test row
PSEUDO_WEIGHT_BASE   = 0.30    # bobot maksimum sample dari pseudo-label
RUN_OPTUNA           = True    # set False untuk pakai default params saja

TOURNAMENT_WEIGHTS = {"FIFA World Cup": 2.00, "AFC Championship": 1.80,
                      "Friendly": 0.96}
DEFAULT_WEIGHT     = 1.20
ALTITUDE_SENTINEL  = -9999.0


ModuleNotFoundError: No module named 'catboost'

## 2. Loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test  = pd.read_csv(DATA_DIR / "test.csv",          parse_dates=["date"])
sub_template = pd.read_csv(DATA_DIR / "sample submission.csv")

train = train.sort_values("date").reset_index(drop=True)
test  = test.sort_values("date").reset_index(drop=True)
print(train.shape, test.shape)


## 3. Metrik Evaluasi (AW-MAE)

Definisi metrik kompetisi: MAE rata-rata kedua sisi gol, ditambah penalti (1 - exact, 1 - outcome benar, 1 - goal-difference benar) dengan multiplier 1.5 jika outcome salah, dipangkatkan 1.5, lalu dirata-ratakan menggunakan bobot turnamen. 

In [ ]:
def get_tournament_weights(tournaments):
    return pd.Series(tournaments).map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT).to_numpy(dtype=float)

def aw_mae(team_true, opp_true, team_pred, opp_pred, tournaments):
    tt = np.asarray(team_true, float); ot = np.asarray(opp_true, float)
    tp = np.asarray(team_pred, float); op = np.asarray(opp_pred, float)
    mae = (np.abs(tt - tp) + np.abs(ot - op)) / 2.0
    exact   = ((tp == tt) & (op == ot)).astype(float)
    outcome = (np.sign(tp - op) == np.sign(tt - ot)).astype(float)
    gd      = ((tp - op) == (tt - ot)).astype(float)
    pen     = 0.30*(1-exact) + 0.25*(1-outcome) + 0.15*(1-gd)
    mult    = np.where(outcome == 1.0, 1.0, 1.5)
    loss    = ((mae + pen) * mult) ** 1.5
    w       = get_tournament_weights(tournaments)
    return float((loss * w).sum() / w.sum())

def aw_mae_round(tt, ot, tp_c, op_c, tournaments, dt=0.0, do=0.0):
    tp = np.maximum(0, np.round(np.asarray(tp_c) + dt)).astype(int)
    op = np.maximum(0, np.round(np.asarray(op_c) + do)).astype(int)
    return aw_mae(tt, ot, tp, op, tournaments)

assert abs(aw_mae([2,1,0],[1,1,2],[2,1,0],[1,1,2],["Friendly"]*3)) < 1e-9


## 4. Feature Engineering

Sebagian besar fitur rolling sudah tersedia di `train.csv`. Untuk
test, kolom-kolom tersebut tidak ada, sehingga direkonstruksi murni dari
riwayat train.

In [ ]:
# Rename kolom panjang ke alias singkat agar kode di bawah ringkas.
RENAME = {
    "team_points_last5": "team_pts_l5", "opp_points_last5": "opp_pts_l5",
    "team_gd_last5": "team_gd_l5",      "opp_gd_last5": "opp_gd_l5",
    "team_points_last10": "team_pts_l10", "opp_points_last10": "opp_pts_l10",
    "team_avg_goals_last5": "team_ag_l5", "team_avg_conceded_last5": "team_ac_l5",
    "opp_avg_goals_last5":  "opp_ag_l5",  "opp_avg_conceded_last5":  "opp_ac_l5",
    "team_win_rate_last10": "team_wr_l10","opp_win_rate_last10": "opp_wr_l10",
    "days_since_last_match_team": "team_days_since",
    "days_since_last_match_opp":  "opp_days_since",
    "h2h_points_last5": "h2h_pts_l5", "h2h_gd_last5": "h2h_gd_l5",
    "elo_team": "elo_team_orig", "elo_opponent": "elo_opp_orig",
    "rank_team": "fifa_rank_team", "rank_opponent": "fifa_rank_opp",
}
train = train.rename(columns=RENAME)
test  = test.rename(columns=RENAME)


In [ ]:
# Career & H2H aggregator: dibangun chronologically dari train.

def build_career_features(train_df, test_df, decay_lambda=0.005):
    # Decay exponential: bobot e^(-lambda * days_ago). Tradisi rating
    # sepak bola (Hvattum & Arntzen 2010) menunjukkan recency penting.
    cols_init = ["team_career_gf", "team_career_ga", "team_career_n",
                 "team_decay_gf", "team_decay_ga", "team_decay_n",
                 "opp_career_gf", "opp_career_ga", "opp_career_n",
                 "opp_decay_gf",  "opp_decay_ga",  "opp_decay_n",
                 "h2h_count", "h2h_team_wins"]
    for c in cols_init:
        train_df[c] = 0.0
        test_df[c]  = 0.0

    state = {}  # team -> dict(gf,ga,n, dgf,dga,dn, last_date)
    h2h   = {}  # frozenset({a,b}) -> dict(count, wins_for_a, a_name)

    def get_team(t):
        if t not in state:
            state[t] = dict(gf=0.0, ga=0.0, n=0, dgf=0.0, dga=0.0, dn=0.0,
                            last=None)
        return state[t]

    def decay_factor(prev_date, cur_date):
        if prev_date is None:
            return 1.0
        d = (cur_date - prev_date).days
        return math.exp(-decay_lambda * max(d, 0))

    train_df = train_df.sort_values(["date", "match_id"]).reset_index(drop=True)
    train_idx = train_df.index.to_numpy()

    # Pass 1: assign features ke setiap baris train berdasarkan state
    for i in train_idx:
        row = train_df.iloc[i]
        t, o, d = row["team"], row["opponent"], row["date"]
        gf, ga = row["team_goals"], row["opp_goals"]

        st = get_team(t); so = get_team(o)
        df_t = decay_factor(st["last"], d); df_o = decay_factor(so["last"], d)

        train_df.at[i, "team_career_gf"] = st["gf"] / max(st["n"], 1)
        train_df.at[i, "team_career_ga"] = st["ga"] / max(st["n"], 1)
        train_df.at[i, "team_career_n"]  = st["n"]
        train_df.at[i, "team_decay_gf"]  = (st["dgf"] * df_t) / max(st["dn"] * df_t, 1e-6)
        train_df.at[i, "team_decay_ga"]  = (st["dga"] * df_t) / max(st["dn"] * df_t, 1e-6)
        train_df.at[i, "team_decay_n"]   = st["dn"] * df_t
        train_df.at[i, "opp_career_gf"]  = so["gf"] / max(so["n"], 1)
        train_df.at[i, "opp_career_ga"]  = so["ga"] / max(so["n"], 1)
        train_df.at[i, "opp_career_n"]   = so["n"]
        train_df.at[i, "opp_decay_gf"]   = (so["dgf"] * df_o) / max(so["dn"] * df_o, 1e-6)
        train_df.at[i, "opp_decay_ga"]   = (so["dga"] * df_o) / max(so["dn"] * df_o, 1e-6)
        train_df.at[i, "opp_decay_n"]    = so["dn"] * df_o

        key = frozenset({t, o})
        h = h2h.get(key, dict(count=0, wins_t=0, t_name=t))
        train_df.at[i, "h2h_count"] = h["count"]
        train_df.at[i, "h2h_team_wins"] = (h["wins_t"] if h["t_name"] == t
                                           else (h["count"] - h["wins_t"]))

        # Update state
        st["gf"]  += gf; st["ga"] += ga; st["n"] += 1
        st["dgf"] = st["dgf"] * df_t + gf
        st["dga"] = st["dga"] * df_t + ga
        st["dn"]  = st["dn"]  * df_t + 1.0
        st["last"] = d
        so["gf"]  += ga; so["ga"] += gf; so["n"] += 1
        so["dgf"] = so["dgf"] * df_o + ga
        so["dga"] = so["dga"] * df_o + gf
        so["dn"]  = so["dn"]  * df_o + 1.0
        so["last"] = d
        if h["count"] == 0:
            h["t_name"] = t
        h["count"] += 1
        if gf > ga:
            if h["t_name"] == t: h["wins_t"] += 1
        h2h[key] = h

    # Pass 2: assign features ke test pakai state akhir train
    for i in test_df.index:
        row = test_df.iloc[i]
        t, o, d = row["team"], row["opponent"], row["date"]
        st = state.get(t, None); so = state.get(o, None)
        if st is not None:
            df_t = decay_factor(st["last"], d)
            test_df.at[i, "team_career_gf"] = st["gf"] / max(st["n"], 1)
            test_df.at[i, "team_career_ga"] = st["ga"] / max(st["n"], 1)
            test_df.at[i, "team_career_n"]  = st["n"]
            denom = max(st["dn"] * df_t, 1e-6)
            test_df.at[i, "team_decay_gf"]  = (st["dgf"] * df_t) / denom
            test_df.at[i, "team_decay_ga"]  = (st["dga"] * df_t) / denom
            test_df.at[i, "team_decay_n"]   = st["dn"] * df_t
        if so is not None:
            df_o = decay_factor(so["last"], d)
            test_df.at[i, "opp_career_gf"]  = so["gf"] / max(so["n"], 1)
            test_df.at[i, "opp_career_ga"]  = so["ga"] / max(so["n"], 1)
            test_df.at[i, "opp_career_n"]   = so["n"]
            denom = max(so["dn"] * df_o, 1e-6)
            test_df.at[i, "opp_decay_gf"]   = (so["dgf"] * df_o) / denom
            test_df.at[i, "opp_decay_ga"]   = (so["dga"] * df_o) / denom
            test_df.at[i, "opp_decay_n"]    = so["dn"] * df_o
        key = frozenset({t, o})
        h = h2h.get(key, None)
        if h is not None:
            test_df.at[i, "h2h_count"] = h["count"]
            test_df.at[i, "h2h_team_wins"] = (h["wins_t"] if h["t_name"] == t
                                               else (h["count"] - h["wins_t"]))
    return train_df, test_df

t0 = time.time()
train, test = build_career_features(train, test)
print(f"Career features built in {time.time()-t0:.1f}s")


In [ ]:
def add_features(df):
    df = df.copy()
    df["year"]      = df["date"].dt.year
    df["month"]     = df["date"].dt.month
    df["dow"]       = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["doy_sin"]   = np.sin(2*np.pi*df["dayofyear"]/365.25)
    df["doy_cos"]   = np.cos(2*np.pi*df["dayofyear"]/365.25)
    df["decade"]    = (df["year"] // 10) * 10

    # Elo derivatives
    elo_t = df.get("elo_team_orig", pd.Series([1500.0]*len(df))).fillna(1500.0)
    elo_o = df.get("elo_opp_orig",  pd.Series([1500.0]*len(df))).fillna(1500.0)
    df["elo_team"]         = elo_t
    df["elo_opp"]          = elo_o
    df["elo_diff"]         = elo_t - elo_o
    df["elo_sum"]          = elo_t + elo_o
    df["elo_diff_sq"]      = df["elo_diff"] ** 2
    df["elo_expected_team"] = 1.0 / (1.0 + 10 ** (-df["elo_diff"] / 400.0))

    # Stakes
    def classify(t):
        if pd.isna(t): return 1
        s = str(t).lower()
        if "friendly" in s: return 0
        if "qualif"   in s: return 2
        if any(k in s for k in ["world cup", "championship", "euro", "copa"]):
            return 3
        return 1
    df["stakes"]      = df["tournament"].apply(classify)
    df["tour_weight"] = get_tournament_weights(df["tournament"])

    df["is_home_eff"] = df["is_home"] * (1 - df["neutral"])

    # Geografis log-scale (right-skewed). Sentinel altitude diganti NaN.
    for c in ["population_team", "population_opp",
              "gdp_per_capita_team", "gdp_per_capita_opp"]:
        if c in df:
            df["log_" + c] = np.log1p(df[c].clip(lower=0).fillna(0))
    df["altitude_clean"] = df["altitude_venue"].replace(ALTITUDE_SENTINEL, np.nan)
    if "distance_travel_team" in df and "distance_travel_opp" in df:
        df["distance_diff"] = df["distance_travel_team"] - df["distance_travel_opp"]
    df["gdp_diff"] = df.get("log_gdp_per_capita_team", 0) - df.get("log_gdp_per_capita_opp", 0)
    df["pop_diff"] = df.get("log_population_team",   0) - df.get("log_population_opp",   0)

    # Diff fitur form
    for a, b, name in [("team_pts_l5", "opp_pts_l5", "pts_l5_diff"),
                       ("team_gd_l5",  "opp_gd_l5",  "gd_l5_diff"),
                       ("team_pts_l10","opp_pts_l10","pts_l10_diff"),
                       ("team_wr_l10", "opp_wr_l10", "wr_l10_diff"),
                       ("team_ag_l5",  "opp_ag_l5",  "ag_l5_diff"),
                       ("team_ac_l5",  "opp_ac_l5",  "ac_l5_diff")]:
        if a in df and b in df:
            df[name] = df[a].fillna(0) - df[b].fillna(0)

    # Career derivatives
    df["career_gf_diff"]        = df["team_career_gf"] - df["opp_career_gf"]
    df["career_ga_diff"]        = df["team_career_ga"] - df["opp_career_ga"]
    df["decay_gf_diff"]         = df["team_decay_gf"]  - df["opp_decay_gf"]
    df["team_attack_vs_opp_def"] = df["team_career_gf"] - df["opp_career_ga"]
    df["opp_attack_vs_team_def"] = df["opp_career_gf"] - df["team_career_ga"]
    df["team_attack_vs_opp_def_decay"] = df["team_decay_gf"] - df["opp_decay_ga"]
    df["opp_attack_vs_team_def_decay"] = df["opp_decay_gf"] - df["team_decay_ga"]
    df["h2h_team_winrate"] = df["h2h_team_wins"] / df["h2h_count"].replace(0, np.nan)
    df["h2h_team_winrate"] = df["h2h_team_winrate"].fillna(0.5)

    # Form x Elo (interaksi)
    df["form_x_elo"]        = (df.get("pts_l5_diff", 0) / 15.0) * (df["elo_diff"] / 400.0)
    df["elo_diff_x_home"]   = df["elo_diff"] * df["is_home_eff"]
    df["elo_diff_x_stakes"] = df["elo_diff"] * df["stakes"]

    # Missing flags untuk fitur rolling/elo
    for c in ["team_pts_l5","opp_pts_l5","team_gd_l5","opp_gd_l5",
              "team_pts_l10","opp_pts_l10","team_ag_l5","team_ac_l5",
              "opp_ag_l5","opp_ac_l5","team_wr_l10","opp_wr_l10",
              "h2h_pts_l5","h2h_gd_l5","fifa_rank_team","fifa_rank_opp"]:
        if c in df:
            df[c + "_isnull"] = df[c].isna().astype(np.int8)

    return df

train_f = add_features(train)
test_f  = add_features(test)
print(train_f.shape, test_f.shape)


In [ ]:
# Encoding ordinal frequency-based untuk kolom kategorikal
def freq_ordinal(train_series, test_series):
    counts = train_series.fillna("__NA__").value_counts()
    mapping = {v: i for i, v in enumerate(counts.index)}
    return (train_series.fillna("__NA__").map(mapping).astype(np.int32),
            test_series.fillna("__NA__").map(mapping).fillna(-1).astype(np.int32))

CAT_COLS = ["tournament", "venue_country", "confederation_team",
            "confederation_opp", "gender"]
for c in CAT_COLS:
    train_f[c + "_enc"], test_f[c + "_enc"] = freq_ordinal(train_f[c], test_f[c])

EXCLUDE = set(["Id", "match_id", "date", "team", "opponent",
               "team_goals", "opp_goals", "tournament", "venue_country",
               "confederation_team", "confederation_opp", "gender",
               "tour_weight", "rank_missing_team", "rank_missing_opp",
               "rank_diff", "fifa_rank_team", "fifa_rank_opp"])

FEATURES = [c for c in train_f.columns
            if c not in EXCLUDE and pd.api.types.is_numeric_dtype(train_f[c])]

for c in FEATURES:
    if c not in test_f.columns:
        test_f[c] = np.nan
print(f"Total fitur: {len(FEATURES)}")


## 5. Cross-Validation Time-Based

Pembagian fold berdasarkan tanggal pertandingan agar tidak ada bocoran
masa depan. Kedua baris dari satu match selalu jatuh di fold yang sama
(group-by `match_id`).

In [ ]:
def make_time_match_folds(df, n_splits=5):
    md = df.groupby("match_id")["date"].min().sort_values()
    ids = md.index.values
    n = len(ids)
    folds_arr = np.minimum((np.arange(n) * n_splits // n), n_splits - 1)
    fmap = dict(zip(ids, folds_arr))
    fa = df["match_id"].map(fmap).values
    out = []
    for k in range(n_splits):
        tr = np.where(fa < k)[0]
        va = np.where(fa == k)[0]
        if len(tr) >= 1000:
            out.append((tr, va))
    return out, fa

train_f = train_f.sort_values("date").reset_index(drop=True)
folds, fold_arr = make_time_match_folds(train_f, N_FOLDS)
print(f"Folds usable: {len(folds)}")
for k, (tr, va) in enumerate(folds):
    d0 = train_f.iloc[va]["date"].min().date()
    d1 = train_f.iloc[va]["date"].max().date()
    print(f"  Fold {k+1}: train={len(tr)}, val={len(va)}, range={d0} -> {d1}")

y_team = train_f["team_goals"].values.astype(float)
y_opp  = train_f["opp_goals"].values.astype(float)
X_train_full = train_f[FEATURES].replace([np.inf, -np.inf], np.nan).copy()
X_test_full  = test_f[FEATURES].replace([np.inf, -np.inf], np.nan).copy()

sample_w = train_f["tournament"].apply(
    lambda x: TOURNAMENT_WEIGHTS.get(x, DEFAULT_WEIGHT)).values

oof_mask = np.zeros(len(train_f), dtype=bool)
for _, va in folds: oof_mask[va] = True
tour_arr_oof = train_f.loc[oof_mask, "tournament"].values


## 6. Target Encoding OOF-Safe

Mean-encoding turnamen dan venue terhadap target gol, dihitung khusus
di dalam tiap fold (hanya pakai fold-fold lampau) supaya tidak bocor.

In [ ]:
def add_target_enc(X_tr, y, X_va, X_te, key_col, alpha=20.0):
    # Encoding bayesian smoothing: (sum + alpha*global)/(count + alpha)
    g = float(np.mean(y))
    df = pd.DataFrame({"k": key_col, "y": y})
    agg = df.groupby("k")["y"].agg(["sum", "count"])
    enc = (agg["sum"] + alpha * g) / (agg["count"] + alpha)
    X_va_enc = X_va.map(enc).fillna(g).values
    X_te_enc = X_te.map(enc).fillna(g).values
    return X_va_enc, X_te_enc


## 7. Optuna Tuning

Pencarian hyperparameter intensif per (model, target). Objective adalah
AW-MAE pada OOF dengan rounding optimal sederhana (offset 0).

In [ ]:
def cv_score_lgbm(params, X, y, sw, folds, objective):
    oof = np.full(len(X), np.nan)
    p = dict(params); p["objective"] = objective; p["metric"] = "mae"
    p["verbose"] = -1
    for tr, va in folds:
        dtr = lgb.Dataset(X.iloc[tr], y[tr], weight=sw[tr])
        dva = lgb.Dataset(X.iloc[va], y[va], weight=sw[va], reference=dtr)
        m = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(80, verbose=False)])
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
    return oof

def cv_score_xgb(params, X, y, sw, folds):
    oof = np.full(len(X), np.nan)
    p = dict(params); p["objective"] = "count:poisson"
    p["eval_metric"] = "mae"; p["tree_method"] = "hist"; p["verbosity"] = 0
    for tr, va in folds:
        dtr = xgb.DMatrix(X.iloc[tr], label=y[tr], weight=sw[tr])
        dva = xgb.DMatrix(X.iloc[va], label=y[va], weight=sw[va])
        m = xgb.train(p, dtr, num_boost_round=3000, evals=[(dva, "v")],
                      early_stopping_rounds=80, verbose_eval=False)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
    return oof

def cv_score_cat(params, X, y, sw, folds):
    oof = np.full(len(X), np.nan)
    p = dict(params); p["loss_function"] = "MAE"; p["verbose"] = False
    for tr, va in folds:
        m = CatBoostRegressor(**p, iterations=3000, early_stopping_rounds=80)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw[tr],
              eval_set=(X.iloc[va], y[va]), use_best_model=True)
        oof[va] = m.predict(X.iloc[va])
    return oof

def oof_awmae(oof_t, oof_o):
    return aw_mae_round(y_team[oof_mask], y_opp[oof_mask],
                        oof_t[oof_mask], oof_o[oof_mask], tour_arr_oof)


In [ ]:
def tune_model(model_kind, target_name, n_trials):
    y = y_team if target_name == "team" else y_opp
    sampler = TPESampler(seed=MASTER_SEED + hash(model_kind + target_name) % 1000,
                         n_startup_trials=20)
    study = optuna.create_study(direction="minimize", sampler=sampler)

    def lgbm_space(trial, poisson_obj):
        p = dict(
            learning_rate    = trial.suggest_float("lr", 0.01, 0.08, log=True),
            num_leaves       = trial.suggest_int("nl", 31, 1024, log=True),
            min_data_in_leaf = trial.suggest_int("mdl", 1, 200, log=True),
            feature_fraction = trial.suggest_float("ff", 0.6, 1.0),
            bagging_fraction = trial.suggest_float("bf", 0.6, 1.0),
            bagging_freq     = 1,
            lambda_l1        = trial.suggest_float("l1", 1e-8, 5.0, log=True),
            lambda_l2        = trial.suggest_float("l2", 1e-8, 5.0, log=True),
            seed             = MASTER_SEED,
        )
        if poisson_obj:
            p["poisson_max_delta_step"] = trial.suggest_float("pmd", 0.1, 1.5)
        return p

    def objective(trial):
        if model_kind == "lgbm_poisson":
            p = lgbm_space(trial, True)
            oof = cv_score_lgbm(p, X_train_full, y, sample_w, folds, "poisson")
        elif model_kind == "lgbm_mae":
            p = lgbm_space(trial, False)
            oof = cv_score_lgbm(p, X_train_full, y, sample_w, folds, "regression_l1")
        elif model_kind == "xgb_poisson":
            p = dict(
                learning_rate=trial.suggest_float("lr", 0.01, 0.08, log=True),
                max_depth=trial.suggest_int("md", 6, 14),
                min_child_weight=trial.suggest_int("mcw", 1, 20),
                subsample=trial.suggest_float("ss", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("cs", 0.6, 1.0),
                reg_alpha=trial.suggest_float("ra", 1e-8, 5.0, log=True),
                reg_lambda=trial.suggest_float("rl", 1e-8, 5.0, log=True),
                seed=MASTER_SEED,
            )
            oof = cv_score_xgb(p, X_train_full, y, sample_w, folds)
        else:  # catboost_mae
            p = dict(
                learning_rate=trial.suggest_float("lr", 0.01, 0.08, log=True),
                depth=trial.suggest_int("d", 6, 12),
                l2_leaf_reg=trial.suggest_float("l2", 1.0, 10.0),
                random_strength=trial.suggest_float("rs", 0.0, 5.0),
                bagging_temperature=trial.suggest_float("bt", 0.0, 1.0),
                random_seed=MASTER_SEED,
            )
            oof = cv_score_cat(p, X_train_full, y, sample_w, folds)
        # Score pakai pasangan target lain pakai mean train sebagai dummy supaya AW-MAE tetap terdefinisi 
        dummy = np.full_like(oof, np.mean(y_team if target_name == "opp" else y_opp))
        if target_name == "team":
            return oof_awmae(oof, dummy)
        else:
            return oof_awmae(dummy, oof)

    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value

BEST_PARAMS = {}
if RUN_OPTUNA:
    for kind in ["lgbm_poisson", "lgbm_mae", "xgb_poisson", "catboost_mae"]:
        for tgt in ["team", "opp"]:
            t0 = time.time()
            bp, bv = tune_model(kind, tgt, N_OPTUNA_TRIALS[kind])
            BEST_PARAMS[(kind, tgt)] = bp
            print(f"[{kind}/{tgt}] best={bv:.4f} in {time.time()-t0:.0f}s")
else:
    DEFAULTS = {
        "lgbm_poisson": dict(lr=0.05, nl=255, mdl=20, ff=0.9, bf=0.9, l1=1e-3, l2=1e-3, pmd=0.7),
        "lgbm_mae":     dict(lr=0.05, nl=255, mdl=20, ff=0.9, bf=0.9, l1=1e-3, l2=1e-3),
        "xgb_poisson":  dict(lr=0.05, md=10,  mcw=5, ss=0.9, cs=0.9, ra=1e-3, rl=1e-3),
        "catboost_mae": dict(lr=0.05, d=8,    l2=3.0, rs=1.0, bt=0.5),
    }
    for kind in DEFAULTS:
        for tgt in ["team", "opp"]:
            BEST_PARAMS[(kind, tgt)] = DEFAULTS[kind]
print("Best params siap.")


## 8. Bagging Multi-Seed + OOF

Untuk tiap (model, target), satu pass CV lengkap (untuk dapat OOF
prediksi) lalu retrain pada full data dengan banyak seed berbeda. Hasil
test prediksi adalah rata-rata dari seluruh seed. 

In [ ]:
def expand_lgbm(p, poisson_obj):
    out = dict(p)
    out["objective"] = "poisson" if poisson_obj else "regression_l1"
    out["metric"]    = "mae"
    out["verbose"]   = -1
    return out

def cv_then_bag_lgbm(p, X, y, X_te, sw, folds, poisson_obj, n_seeds):
    par = expand_lgbm(p, poisson_obj)
    # CV untuk OOF + best_iter rata-rata
    oof = np.zeros(len(X))
    best_iters = []
    for tr, va in folds:
        dtr = lgb.Dataset(X.iloc[tr], y[tr], weight=sw[tr])
        dva = lgb.Dataset(X.iloc[va], y[va], weight=sw[va], reference=dtr)
        m = lgb.train(par, dtr, num_boost_round=3000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(80, verbose=False)])
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
        best_iters.append(m.best_iteration)
    n_round = int(np.mean(best_iters) * 1.05)
    # Bagging full-data
    test_pred = np.zeros(len(X_te))
    for s in range(n_seeds):
        par_s = dict(par)
        par_s["seed"] = MASTER_SEED + s
        par_s["bagging_seed"] = MASTER_SEED + s * 7
        par_s["feature_fraction_seed"] = MASTER_SEED + s * 13
        dall = lgb.Dataset(X, y, weight=sw)
        m = lgb.train(par_s, dall, num_boost_round=n_round)
        test_pred += m.predict(X_te) / n_seeds
    return oof, test_pred

def cv_then_bag_xgb(p, X, y, X_te, sw, folds, n_seeds):
    par = dict(p); par["objective"] = "count:poisson"
    par["eval_metric"] = "mae"; par["tree_method"] = "hist"; par["verbosity"] = 0
    oof = np.zeros(len(X))
    best_iters = []
    for tr, va in folds:
        dtr = xgb.DMatrix(X.iloc[tr], label=y[tr], weight=sw[tr])
        dva = xgb.DMatrix(X.iloc[va], label=y[va], weight=sw[va])
        m = xgb.train(par, dtr, num_boost_round=3000, evals=[(dva, "v")],
                      early_stopping_rounds=80, verbose_eval=False)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
        best_iters.append(m.best_iteration + 1)
    n_round = int(np.mean(best_iters) * 1.05)
    dall = xgb.DMatrix(X, label=y, weight=sw)
    dtest = xgb.DMatrix(X_te)
    test_pred = np.zeros(len(X_te))
    for s in range(n_seeds):
        par_s = dict(par); par_s["seed"] = MASTER_SEED + s
        m = xgb.train(par_s, dall, num_boost_round=n_round)
        test_pred += m.predict(dtest) / n_seeds
    return oof, test_pred

def cv_then_bag_cat(p, X, y, X_te, sw, folds, n_seeds):
    par = dict(p); par["loss_function"] = "MAE"; par["verbose"] = False
    oof = np.zeros(len(X))
    best_iters = []
    for tr, va in folds:
        m = CatBoostRegressor(**par, iterations=3000, early_stopping_rounds=80)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw[tr],
              eval_set=(X.iloc[va], y[va]), use_best_model=True)
        oof[va] = m.predict(X.iloc[va])
        best_iters.append(m.tree_count_)
    n_round = int(np.mean(best_iters) * 1.05)
    test_pred = np.zeros(len(X_te))
    for s in range(n_seeds):
        par_s = dict(par); par_s["random_seed"] = MASTER_SEED + s
        m = CatBoostRegressor(**par_s, iterations=n_round)
        m.fit(X, y, sample_weight=sw)
        test_pred += m.predict(X_te) / n_seeds
    return oof, test_pred


In [ ]:
def run_full_zoo(X, y_t, y_o, X_te, sw, folds, params_dict, n_seeds):
    out = {}
    for kind in ["lgbm_poisson", "lgbm_mae", "xgb_poisson", "catboost_mae"]:
        for tgt, y in [("team", y_t), ("opp", y_o)]:
            t0 = time.time()
            p = params_dict[(kind, tgt)]
            if kind == "lgbm_poisson":
                oof, tp = cv_then_bag_lgbm(p, X, y, X_te, sw, folds, True,  n_seeds)
            elif kind == "lgbm_mae":
                oof, tp = cv_then_bag_lgbm(p, X, y, X_te, sw, folds, False, n_seeds)
            elif kind == "xgb_poisson":
                oof, tp = cv_then_bag_xgb (p, X, y, X_te, sw, folds, n_seeds)
            else:
                oof, tp = cv_then_bag_cat (p, X, y, X_te, sw, folds, n_seeds)
            out[(kind, tgt, "oof")]  = oof
            out[(kind, tgt, "test")] = tp
            print(f"[{kind}/{tgt}] {time.time()-t0:.0f}s")
    return out

t0 = time.time()
ZOO = run_full_zoo(X_train_full, y_team, y_opp, X_test_full,
                   sample_w, folds, BEST_PARAMS, N_BAG_SEEDS)
print(f"Zoo round 1 selesai dalam {(time.time()-t0)/60:.1f} menit")


## 9. Per-Match Optimal Poisson Rounding

Tabel kerugian (loss table) berukuran $(N+1)^4$ untuk semua kemungkinan
pasangan skor 0..N. Untuk setiap pertandingan, kita asumsikan distribusi
gol mengikuti Poisson($\lambda$) dengan $\lambda$ = prediksi kontinu,
lalu pilih pasangan integer $(g_t, g_o)$ yang meminimalkan ekspektasi
loss. Strategi ini selaras dengan teori decision-theoretic: Bayes
estimator di bawah loss kustom kompetisi.

In [ ]:
def build_loss_table(max_g=MAX_GOAL):
    N = max_g + 1
    table = np.zeros((N, N, N, N))
    for i in range(N):
        for j in range(N):
            for ti in range(N):
                for tj in range(N):
                    mae = (abs(ti - i) + abs(tj - j)) / 2.0
                    exact = 1.0 if (i == ti and j == tj) else 0.0
                    out = 1.0 if np.sign(i - j) == np.sign(ti - tj) else 0.0
                    gd  = 1.0 if (i - j) == (ti - tj) else 0.0
                    pen = 0.30*(1-exact) + 0.25*(1-out) + 0.15*(1-gd)
                    mult = 1.0 if out == 1 else 1.5
                    table[i, j, ti, tj] = ((mae + pen) * mult) ** 1.5
    return table

def optimal_predictions(lam_t, lam_o, max_g=MAX_GOAL, loss_table=None):
    if loss_table is None: loss_table = build_loss_table(max_g)
    N = max_g + 1
    n = len(lam_t)
    lt = np.maximum(0.05, np.asarray(lam_t, float))
    lo = np.maximum(0.05, np.asarray(lam_o, float))
    p_t = np.stack([poisson.pmf(k, lt) for k in range(N)], axis=1)
    p_o = np.stack([poisson.pmf(k, lo) for k in range(N)], axis=1)
    p_t /= p_t.sum(axis=1, keepdims=True)
    p_o /= p_o.sum(axis=1, keepdims=True)
    expected = np.einsum("ijkl,nk,nl->nij", loss_table, p_t, p_o)
    arg = expected.reshape(n, -1).argmin(axis=1)
    return (arg // N).astype(np.int32), (arg % N).astype(np.int32)

LOSS_TABLE = build_loss_table(MAX_GOAL)
print("Loss table siap")


## 10. Blending Simplex pada OOF

Kombinasi linier dari empat model dengan bobot non-negatif yang
berjumlah satu, dipilih via grid search step 0.05 untuk meminimalkan
AW-MAE OOF. Bobot dicari terpisah untuk target team dan opp karena
struktur error keduanya bisa berbeda (home advantage asimetris).

In [ ]:
def simplex_grid(n_models=4, step=20):
    out = []
    def rec(remaining, depth, cur):
        if depth == n_models - 1:
            cur.append(remaining); out.append(tuple(cur)); cur.pop(); return
        for i in range(remaining + 1):
            cur.append(i); rec(remaining - i, depth + 1, cur); cur.pop()
    rec(step, 0, [])
    return [tuple(x / step for x in w) for w in out]

def blend_and_score(zoo, weights_t, weights_o):
    stack_t = np.column_stack([zoo[(k,"team","oof")]  for k in
                               ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    stack_o = np.column_stack([zoo[(k,"opp","oof")]   for k in
                               ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    ts_t = np.column_stack([zoo[(k,"team","test")] for k in
                            ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    ts_o = np.column_stack([zoo[(k,"opp","test")]  for k in
                            ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    bt = stack_t @ np.array(weights_t); bo = stack_o @ np.array(weights_o)
    bt_te = ts_t @ np.array(weights_t); bo_te = ts_o @ np.array(weights_o)
    return bt, bo, bt_te, bo_te

def find_best_blend(zoo):
    cands = simplex_grid(4, 20)
    stack_t = np.column_stack([zoo[(k,"team","oof")]  for k in
                               ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    stack_o = np.column_stack([zoo[(k,"opp","oof")]   for k in
                               ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
    # Search team weights dengan opp default uniform, lalu sebaliknya.
    best_t, sc_t = None, 1e18
    uniform = np.array([0.25]*4)
    bo_default = stack_o @ uniform
    for w in cands:
        bt = stack_t @ np.array(w)
        # Pakai optimal rounding untuk konsistensi dengan post-proc final.
        gt, go = optimal_predictions(bt[oof_mask], bo_default[oof_mask],
                                     MAX_GOAL, LOSS_TABLE)
        s = aw_mae(y_team[oof_mask], y_opp[oof_mask], gt, go, tour_arr_oof)
        if s < sc_t: best_t, sc_t = w, s
    bt_best = stack_t @ np.array(best_t)
    best_o, sc_o = None, 1e18
    for w in cands:
        bo = stack_o @ np.array(w)
        gt, go = optimal_predictions(bt_best[oof_mask], bo[oof_mask],
                                     MAX_GOAL, LOSS_TABLE)
        s = aw_mae(y_team[oof_mask], y_opp[oof_mask], gt, go, tour_arr_oof)
        if s < sc_o: best_o, sc_o = w, s
    return best_t, best_o, sc_o

w_t, w_o, score_blend = find_best_blend(ZOO)
print(f"Bobot team: {np.round(w_t,3)}; bobot opp: {np.round(w_o,3)}; AW-MAE OOF: {score_blend:.4f}")
blend_oof_t, blend_oof_o, blend_te_t, blend_te_o = blend_and_score(ZOO, w_t, w_o)


## 11. Pseudo-Labeling Confidence-Filtered

In [ ]:
def confidence_score(lam_t, lam_o, gt_int, go_int):
    pt = poisson.pmf(gt_int, np.maximum(0.05, lam_t))
    po = poisson.pmf(go_int, np.maximum(0.05, lam_o))
    return pt * po

best_score = score_blend
best_state = dict(blend_te_t=blend_te_t, blend_te_o=blend_te_o,
                  blend_oof_t=blend_oof_t, blend_oof_o=blend_oof_o,
                  w_t=w_t, w_o=w_o, ZOO=ZOO)
no_improve = 0

X_train_pl = X_train_full.copy()
y_team_pl  = y_team.copy()
y_opp_pl   = y_opp.copy()
sw_pl      = sample_w.copy()
tour_train_pl = train_f["tournament"].values.copy()

for rnd in range(1, N_PSEUDO_ROUNDS + 1):
    print(f"\n--- Pseudo round {rnd} ---")
    # Hitung confidence pada test
    gt_te, go_te = optimal_predictions(best_state["blend_te_t"],
                                        best_state["blend_te_o"],
                                        MAX_GOAL, LOSS_TABLE)
    conf = confidence_score(best_state["blend_te_t"],
                             best_state["blend_te_o"], gt_te, go_te)
    thr = np.quantile(conf, PSEUDO_CONF_QUANTILE)
    keep = conf >= thr
    
    rng = np.random.default_rng(MASTER_SEED + rnd * 17)
    keep_idx = np.where(keep)[0]
    sub_idx = rng.choice(keep_idx, size=int(0.8 * len(keep_idx)), replace=False)
    print(f"  threshold conf={thr:.4f}, pseudo rows={len(sub_idx)}")

    X_pseudo = X_test_full.iloc[sub_idx].copy()
    # Soft label: pakai prediksi kontinu (bukan integer) supaya model bisa belajar tendency.
    y_t_pseudo = best_state["blend_te_t"][sub_idx]
    y_o_pseudo = best_state["blend_te_o"][sub_idx]
    conf_norm = (conf[sub_idx] - conf[sub_idx].min()) / (conf[sub_idx].ptp() + 1e-9)
    sw_pseudo = PSEUDO_WEIGHT_BASE * conf_norm * \
                np.array([TOURNAMENT_WEIGHTS.get(t, DEFAULT_WEIGHT)
                          for t in test_f.iloc[sub_idx]["tournament"].values])

    X_combined = pd.concat([X_train_pl, X_pseudo], ignore_index=True)
    y_t_combined = np.concatenate([y_team_pl, y_t_pseudo])
    y_o_combined = np.concatenate([y_opp_pl,  y_o_pseudo])
    sw_combined  = np.concatenate([sw_pl, sw_pseudo])

    # Rebuild folds untuk ukuran baru: pseudo rows masuk fold terakhir agar tidak mempengaruhi val original.
    base_n = len(X_train_pl)
    new_folds = []
    for tr, va in folds:
        new_folds.append((np.concatenate([tr, np.arange(base_n, len(X_combined))]), va))

    ZOO_R = run_full_zoo(X_combined, y_t_combined, y_o_combined,
                         X_test_full, sw_combined, new_folds, BEST_PARAMS,
                         max(20, N_BAG_SEEDS // 2))
    w_t_r, w_o_r, sc_r = find_best_blend(ZOO_R)
    bt_oof_r, bo_oof_r, bt_te_r, bo_te_r = blend_and_score(ZOO_R, w_t_r, w_o_r)
    print(f"  AW-MAE OOF round {rnd}: {sc_r:.4f} (best={best_score:.4f})")

    if sc_r < best_score - 1e-4:
        best_score = sc_r
        best_state = dict(blend_te_t=bt_te_r, blend_te_o=bo_te_r,
                          blend_oof_t=bt_oof_r, blend_oof_o=bo_oof_r,
                          w_t=w_t_r, w_o=w_o_r, ZOO=ZOO_R)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= 2:
            print("  Stop: tidak ada peningkatan dua ronde.")
            break

ZOO_FINAL = best_state["ZOO"]
blend_oof_t = best_state["blend_oof_t"]; blend_oof_o = best_state["blend_oof_o"]
blend_te_t  = best_state["blend_te_t"];  blend_te_o  = best_state["blend_te_o"]
print(f"\nFinal AW-MAE OOF: {best_score:.4f}")


## 12. Submission

In [ ]:
# Cari offset rounding terbaik (grid tipis) sebagai sanity-check tambahan di samping optimal Poisson rounding.
best_off = (0.0, 0.0, 1e18)
for dt_ in np.arange(-0.3, 0.31, 0.05):
    for do_ in np.arange(-0.3, 0.31, 0.05):
        s = aw_mae_round(y_team[oof_mask], y_opp[oof_mask],
                         blend_oof_t[oof_mask], blend_oof_o[oof_mask],
                         tour_arr_oof, dt_, do_)
        if s < best_off[2]: best_off = (dt_, do_, s)
print(f"Offset terbaik: dt={best_off[0]:.2f}, do={best_off[1]:.2f}, AW-MAE={best_off[2]:.4f}")

# Pakai optimal Poisson untuk submission.
gt_te, go_te = optimal_predictions(blend_te_t, blend_te_o, MAX_GOAL, LOSS_TABLE)

sub = pd.DataFrame({"Id": test_f["Id"].values,
                    "team_goals": gt_te, "opp_goals": go_te})

# Selaraskan urutan dengan sample submission.
sub = sub_template[["Id"]].merge(sub, on="Id", how="left")
sub.to_csv("submission.csv", index=False)
print(sub.head())
print(f"submission.csv ditulis ({len(sub)} baris)")


## 13. Evaluation

In [ ]:
# Distribusi prediksi vs distribusi train (KS test)
for col_pred, col_true, name in [(gt_te, y_team, "team_goals"),
                                  (go_te, y_opp,  "opp_goals")]:
    ks = ks_2samp(col_pred, col_true)
    print(f"KS {name}: stat={ks.statistic:.4f}, p={ks.pvalue:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (pred, true, name) in zip(axes, [(gt_te, y_team, "team_goals"),
                                          (go_te, y_opp,  "opp_goals")]):
    bins = np.arange(-0.5, MAX_GOAL + 1.5, 1)
    ax.hist(true, bins=bins, alpha=0.5, density=True, label="train")
    ax.hist(pred, bins=bins, alpha=0.5, density=True, label="pred test")
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Calibration plot Poisson
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, lam, y, name in [(axes[0], blend_oof_t[oof_mask], y_team[oof_mask], "team"),
                         (axes[1], blend_oof_o[oof_mask], y_opp[oof_mask],  "opp")]:
    bins = np.linspace(0, max(4, np.quantile(lam, 0.99)), 11)
    idx = np.digitize(lam, bins)
    xs, ys = [], []
    for b in range(1, len(bins)):
        m = idx == b
        if m.sum() > 30:
            xs.append(lam[m].mean()); ys.append(y[m].mean())
    ax.plot(xs, ys, "o-", label="observed"); ax.plot(xs, xs, "k--", label="ideal")
    ax.set_xlabel("lambda predicted"); ax.set_ylabel("mean observed")
    ax.set_title(f"Calibration {name}"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Outcome distribution
def outcome(t, o):
    s = np.sign(np.asarray(t) - np.asarray(o))
    return pd.Series(s).value_counts(normalize=True).sort_index()

print("Outcome train  :", outcome(y_team, y_opp).round(3).to_dict())
print("Outcome predict:", outcome(gt_te, go_te).round(3).to_dict())

# Goal-difference distribution
plt.figure(figsize=(7, 3.5))
plt.hist(np.abs(y_team - y_opp), bins=np.arange(-0.5, 11), alpha=0.5,
         density=True, label="train")
plt.hist(np.abs(gt_te - go_te), bins=np.arange(-0.5, 11), alpha=0.5,
         density=True, label="pred test")
plt.legend(); plt.title("|GD| distribution"); plt.show()


In [ ]:
# Per-tournament prediction stats (turnamen mayor saja)
focus = ["FIFA World Cup", "UEFA Euro", "Copa America", "AFC Asian Cup",
         "African Cup of Nations", "Friendly"]
rows = []
for t in focus:
    m_tr = train_f["tournament"] == t
    m_te = test_f["tournament"] == t
    if m_te.sum() == 0: continue
    rows.append(dict(tournament=t,
                     n_train=int(m_tr.sum()), n_test=int(m_te.sum()),
                     mean_train=(y_team[m_tr] + y_opp[m_tr]).mean()/1
                                if m_tr.sum() else np.nan,
                     mean_pred=(gt_te[m_te] + go_te[m_te]).mean()))
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# Uncertainty antar seed
stack_te_t = np.column_stack([ZOO_FINAL[(k,"team","test")] for k in
                              ["lgbm_poisson","lgbm_mae","xgb_poisson","catboost_mae"]])
unc = stack_te_t.std(axis=1)
plt.figure(figsize=(7, 3.5))
plt.hist(unc, bins=40); plt.title("Inter-model std (team_goals predictions)")
plt.xlabel("std"); plt.show()
print(f"Uncertainty mean={unc.mean():.3f}, p90={np.quantile(unc,0.9):.3f}")


In [ ]:
# Feature importance dari LGBM-Poisson (team)
par = expand_lgbm(BEST_PARAMS[("lgbm_poisson","team")], True)
par["seed"] = MASTER_SEED
m_imp = lgb.train(par, lgb.Dataset(X_train_full, y_team, weight=sample_w),
                  num_boost_round=400)
imp = pd.Series(m_imp.feature_importance(importance_type="gain"),
                 index=FEATURES).sort_values(ascending=False)
print(imp.head(30))
plt.figure(figsize=(8, 7))
imp.head(30)[::-1].plot.barh(); plt.title("Top 30 feature importance (gain)")
plt.tight_layout(); plt.show()


In [ ]:
# Permutation importance pada OOF
top_feats = imp.head(15).index.tolist()
base_score = aw_mae(y_team[oof_mask], y_opp[oof_mask],
                    *optimal_predictions(blend_oof_t[oof_mask],
                                          blend_oof_o[oof_mask],
                                          MAX_GOAL, LOSS_TABLE),
                    tour_arr_oof)
deltas = {}
rng = np.random.default_rng(MASTER_SEED + 999)
for f in top_feats:
    Xp = X_train_full.copy()
    perm = rng.permutation(len(Xp))
    Xp[f] = Xp[f].values[perm]
    # Refit cepat hanya LGBM-Poisson team untuk efisiensi (proxy importance).
    par = expand_lgbm(BEST_PARAMS[("lgbm_poisson","team")], True)
    par["seed"] = MASTER_SEED
    m = lgb.train(par, lgb.Dataset(Xp, y_team, weight=sample_w), num_boost_round=300)
    pred = m.predict(Xp)
    gt_p, go_p = optimal_predictions(pred[oof_mask], blend_oof_o[oof_mask],
                                      MAX_GOAL, LOSS_TABLE)
    s = aw_mae(y_team[oof_mask], y_opp[oof_mask], gt_p, go_p, tour_arr_oof)
    deltas[f] = s - base_score
print(pd.Series(deltas).sort_values(ascending=False))


In [ ]:
# Residual analysis OOF: scatter pred vs actual + residual vs elo_diff
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gt_oof, go_oof = optimal_predictions(blend_oof_t[oof_mask], blend_oof_o[oof_mask],
                                      MAX_GOAL, LOSS_TABLE)
axes[0].scatter(blend_oof_t[oof_mask], y_team[oof_mask], s=3, alpha=0.3)
axes[0].plot([0, 8], [0, 8], "r--"); axes[0].set_xlabel("pred lambda team")
axes[0].set_ylabel("actual"); axes[0].set_title("OOF team")

resid = y_team[oof_mask] - blend_oof_t[oof_mask]
elo_d = train_f.loc[oof_mask, "elo_diff"].values if "elo_diff" in train_f.columns \
        else train_f.loc[oof_mask, "elo_team"].values - train_f.loc[oof_mask, "elo_opp"].values
axes[1].scatter(elo_d, resid, s=3, alpha=0.3); axes[1].axhline(0, color="r")
axes[1].set_xlabel("elo_diff"); axes[1].set_ylabel("residual team")
plt.tight_layout(); plt.show()


In [ ]:
# Time-stability
print("AW-MAE per fold (optimal rounding):")
for k, (tr, va) in enumerate(folds):
    gt_f, go_f = optimal_predictions(blend_oof_t[va], blend_oof_o[va],
                                      MAX_GOAL, LOSS_TABLE)
    s = aw_mae(y_team[va], y_opp[va], gt_f, go_f,
               train_f.iloc[va]["tournament"].values)
    d0 = train_f.iloc[va]["date"].min().date()
    d1 = train_f.iloc[va]["date"].max().date()
    print(f"  fold {k+1} ({d0}->{d1}): {s:.4f}")
